# SyncUp v0.5 — projeto de dados com CRISP-DM
Carlos Henrique · Sistemas de Informação · UCL

Compare históricos estendidos do Spotify com estatística descritiva, métricas explicáveis e visualizações. Este notebook usa os módulos da versão final, sem duplicar os cálculos da aplicação.

**Padrão seguro para demonstração:** dados fictícios de duas pessoas, gerados localmente com sementes fixas. Para usar dados próprios, edite a célula de configuração. Execute as células em ordem, na pasta que contém `app.py`. Instale `requirements-dev.txt`; selecione um kernel Python desse ambiente.

**Aviso metodológico:** arquétipos e traços descrevem o recorte de escuta. Não medem personalidade ou afinidade amorosa. Não há modelo preditivo, teste psicológico ou conclusão causal.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from analise import TEMPO_MINIMO_PLAY_MS, EntradaPerfil, comparar, comparar_grupo
from app import criar_interface
from cartao import gerar_cartao
from configuracao import ConfiguracaoCarga
from dados import carregar_historicos
from exportacoes import exportar_playlist
from formatacao import fmt_num, fmt_pct
from graficos import butterfly, mapa_horarios, matriz_grupo, radar_tracos, slope_chart
from metricas import contribuicoes_cosseno, cosseno, jaccard_ponderado
from perfis import AVISO_TRACOS, LIMIAR_FAVORITOS, LIMIAR_MADRUGADA, LIMIAR_REPEAT
from tools.gerar_dados_falsos import Configuracao, gerar_zip

RAIZ = Path.cwd()
if not (RAIZ / 'app.py').is_file():
    raise RuntimeError('Abra o notebook na pasta SyncUp, junto aos módulos Python.')

USAR_DADOS_FICTICIOS = True
ABRIR_INTERFACE = False  # Altere para True somente na execução interativa.
NOMES = ('Pessoa A', 'Pessoa B')
PONDERACAO = 'bruta'
PERIODO_COMUM = True
CONFIG = ConfiguracaoCarga(
    fuso_horario='America/Sao_Paulo',
    musicas_ignoradas=(),
    artistas_ignorados=(),
    deduplicar=True,
    usar_cache_disco=False,
)
print(AVISO_TRACOS)

## 1. Entendimento do negócio

**Problema:** duas pessoas podem compartilhar artistas, mas escutá-los com frequências e horários diferentes. Como comparar esses padrões de forma transparente?

**Objetivo:** identificar repertório comum, diferenças de participação, coincidências temporais e possíveis sugestões de escuta. Público: usuários que conseguem exportar seu histórico estendido do Spotify.

**Critério de sucesso:** dados válidos e rastreáveis, comparação reproduzível, métricas com fórmulas claras e uma apresentação que mostre suas limitações. A validação é computacional/metodológica; não se mede relacionamento, qualidade musical ou satisfação subjetiva.

## 2. Entendimento dos dados

Fonte: histórico estendido de áudio do Spotify, em ZIP/JSON. Campos necessários: `ts`, `ms_played`, `master_metadata_track_name` e `master_metadata_album_artist_name`. Campos como IP, plataforma e `shuffle` são opcionais e não são mantidos na tabela musical.

O loader já aplica validações na leitura; os contadores mostram o que foi descartado. `describe()` resume minutos por execução válida: count, mean, std, min, quartis e max. Para comparar contagens, são considerados plays de pelo menos 30 segundos.

In [ ]:
if USAR_DADOS_FICTICIOS:
    pasta_exemplos = RAIZ / 'exemplos_ficticios'
    arquivos_a = [
        gerar_zip(
            pasta_exemplos / 'a.zip',
            Configuracao(linhas=3000, artistas=120, seed=41, tamanho_bloco=1000),
        )
    ]
    arquivos_b = [
        gerar_zip(
            pasta_exemplos / 'b.zip',
            Configuracao(linhas=3000, artistas=120, seed=97, tamanho_bloco=1000),
        )
    ]
    print('Demonstração com dados fictícios; não representa nenhuma pessoa real.')
else:
    # Uma pessoa por lista; vários arquivos da lista são unidos pelo loader.
    arquivos_a = [RAIZ / 'historico_a.zip']
    arquivos_b = [RAIZ / 'historico_b.zip']

historico_a = carregar_historicos(arquivos_a, configuracao=CONFIG)
historico_b = carregar_historicos(arquivos_b, configuracao=CONFIG)
a, b = historico_a.dados, historico_b.dados
for nome, h in zip(NOMES, (historico_a, historico_b), strict=True):
    print(f'{nome}: {fmt_num(len(h.dados))} execuções válidas; fuso {h.fuso_horario}')
    display(pd.DataFrame([asdict(h.contadores)]))
    display(h.dados[['minutos']].describe())

In [ ]:
# EDA acadêmica com Seaborn: contagens brutas do histórico completo de A.
plays_a = a[a.ms_played >= TEMPO_MINIMO_PLAY_MS]
top_artistas = (
    plays_a.groupby('artista', observed=True)
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename('Reproduções')
    .reset_index()
)
top_artistas['artista'] = top_artistas.artista.astype(object)
figura, eixo = plt.subplots(figsize=(11, 6))
sns.barplot(data=top_artistas, x='Reproduções', y='artista', color='#1db954', ax=eixo)
eixo.set_title('Artistas líderes — A, histórico completo, plays ≥30s')
eixo.set_xlabel('Reproduções (contagem bruta)')
eixo.set_ylabel('Artista')
figura.tight_layout()
plt.show()
plt.close(figura)

## 3. Preparação dos dados

`dados.py` separa música/podcast, descarta durações e datas inválidas, limpa nomes e aplica filtros configuráveis. A deduplicação usa instante UTC + duração + artista + música; não usa apenas a hora local. O cache é opcional e considera SHA-256, filtros e fuso.

O loader converte para o fuso escolhido uma única vez, prepara `hora`, `ano_mes`, `dia_semana`, `data_dia` e usa `category` em artista/música. Todos os `groupby` observam apenas categorias presentes. Os totais descartados mais válidos devem fechar o total recebido.

Neste exemplo, a preparação é ativada pela configuração inicial e executada durante a leitura; abaixo conferimos seus efeitos.

In [ ]:
colunas_temporais = ['hora', 'ano_mes', 'dia_semana', 'data_dia']
display(a[colunas_temporais + ['artista', 'musica']].dtypes.to_frame('Tipo preparado'))
assert a.data.dt.tz is None
assert str(a.data_utc.dt.tz) == 'UTC'
assert all(pd.api.types.is_integer_dtype(a[c]) for c in colunas_temporais)
for h in (historico_a, historico_b):
    c = h.contadores
    descartes = (
        c.podcasts
        + c.ms_invalidos
        + c.datas_invalidas
        + c.metadados_invalidos
        + c.various_artists_descartados
        + c.ignorados
        + c.duplicados
    )
    assert c.recebidos == c.validos + descartes
print('Categorias, datas e contabilidade de descartes verificadas.')

## 4. Modelagem

Aqui, **modelagem** significa construir representações e métricas determinísticas; não treinar uma rede neural ou classificador de personalidade.

- Vetor: contagens por artista. Pesos disponíveis: x (`bruta`), √x (`raiz`) ou ln(1+x) (`log1p`).
- Cosseno: produto escalar dividido pelas normas. Cada artista contribui aᵢbᵢ/(|a|·|b|).
- Jaccard: interseção/união de artistas. Ruzicka: soma dos mínimos/soma dos máximos dos pesos absolutos.
- Entropia normalizada e Gini: distribuição das contagens brutas entre artistas observados; não inferem gêneros.
- Sincronização: artista ou faixa iguais, em UTC, com distância inclusiva ≤1h e sem reutilização de eventos dentro de cada análise.
- Ponte Musical: artistas exclusivos de uma origem sugeridos ao outro perfil. Links são buscas Spotify, sem rede/API na análise.

Os rankings usam plays ≥30s. As horas e o `describe()` incluem todas as execuções válidas. O recorte comum considera cobertura UTC; use históricos completos caso não exista interseção, interpretando coberturas diferentes com cuidado.

In [ ]:
resultado = comparar(
    arquivos_a,
    arquivos_b,
    *NOMES,
    periodo_comum=PERIODO_COMUM,
    configuracao=CONFIG,
    ponderacao=PONDERACAO,
)
met = resultado.metricas
print(
    f'Cosseno: {fmt_pct(met.cosseno)}; Jaccard: {fmt_pct(met.jaccard)}; '
    f'Ruzicka: {fmt_pct(met.jaccard_ponderado)}'
)
print(resultado.aviso)
display(resultado.contribuicoes.head(10))
display(butterfly(resultado))
display(slope_chart(resultado))
display(mapa_horarios(resultado))
display(radar_tracos(resultado))

In [ ]:
regras = pd.DataFrame(
    [
        {
            'Dimensão': 'Favoritos',
            'Proporção': 'Top 2 artistas / plays',
            'Limiar': fmt_pct(LIMIAR_FAVORITOS),
        },
        {
            'Dimensão': 'Repetição',
            'Proporção': 'Top 10 faixas / plays',
            'Limiar': fmt_pct(LIMIAR_REPEAT),
        },
        {
            'Dimensão': 'Madrugada',
            'Proporção': '00h–05h59 / plays',
            'Limiar': fmt_pct(LIMIAR_MADRUGADA),
        },
    ]
)
display(regras)
for perfil in (resultado.perfil_a, resultado.perfil_b):
    print(perfil.nome)
    display(pd.DataFrame(asdict(perfil.arquetipos)).T)
    print(f'Entropia: {fmt_num(perfil.entropia, 4)}; Gini observado: {fmt_num(perfil.gini, 4)}')
print(AVISO_TRACOS)
print('Mesma música/artista em até 1h:', resultado.momentos_musicas.total_pares, 'pares')
display(resultado.momentos_musicas.exemplos.head(10))
display(resultado.ponte)
display(resultado.playlist.head(10))

## 5. Avaliação

Verifique cobertura, tamanho de amostra, descartes, ponderação e duplicações antes de interpretar o score. Perfis proporcionais têm cosseno 1, mas podem ter Ruzicka menor que 1 por diferença de volume. Uma porcentagem alta não identifica o dono do arquivo.

A soma das contribuições deve reproduzir o cosseno, com tolerância de ponto flutuante. A suíte pytest também compara a versão atual com referências salvas da v0.4/Passo 1, sem recriar resultados esperados durante os testes.

**Limitações:** categorias de gêneros/aliases não são inferidas; ausência no histórico não prova desinteresse; coincidências de instantes não provam escuta simultânea durante toda a faixa; o recorte comum não garante completude do histórico; top 10 tende a ter proporção alta em amostras pequenas.

In [ ]:
assert abs(cosseno({'A': 3, 'B': 4}, {'A': 6, 'B': 8}) - 1) < 1e-12
assert cosseno({'A': 1}, {'B': 1}) == 0
assert cosseno({}, {}) is None
assert jaccard_ponderado({'A': 10}, {'A': 1}) == 0.1
contribuicoes = contribuicoes_cosseno({'A': 3, 'B': 4}, {'A': 3, 'B': 4})
assert sum(contribuicoes.values()) == 1
assert abs(resultado.contribuicoes['Contribuição ao cosseno'].sum() - met.cosseno) < 1e-12
display(resultado.diagnostico)
display(resultado.estatisticas)
print('Verificações didáticas concluídas. Execute python -m pytest -q para a suíte completa.')

### Avaliação do Modo Grupo (opcional)

No exemplo fictício, geramos uma terceira pessoa e calculamos a matriz. Todo o grupo usa a mesma interseção global se `PERIODO_COMUM=True`. A API suporta de 3 a 6 perfis e várias partes por perfil. Sem dados fictícios, este exemplo é ignorado: configure suas próprias entradas explicitamente.

In [ ]:
if USAR_DADOS_FICTICIOS:
    arquivo_c = gerar_zip(
        pasta_exemplos / 'c.zip',
        Configuracao(linhas=3000, artistas=120, seed=113, tamanho_bloco=1000),
    )
    grupo = comparar_grupo(
        [
            EntradaPerfil(NOMES[0], arquivos_a),
            EntradaPerfil(NOMES[1], arquivos_b),
            EntradaPerfil('Pessoa C', arquivo_c),
        ],
        periodo_comum=PERIODO_COMUM,
        configuracao=CONFIG,
        ponderacao=PONDERACAO,
    )
    display(grupo.matriz)
    display(matriz_grupo(grupo))
else:
    print('Exemplo de grupo não executado. Use EntradaPerfil para fornecer 3–6 perfis.')

## 6. Implantação

Uso local pelo notebook ou interface Gradio 5, com plots interativos Plotly. CSV e cartão são gerados localmente; não existe publicação automática, login ou criação de playlist na conta Spotify.

O CSV inclui todas as faixas comuns, usa UTF-8 BOM e `;`, e protege nomes que poderiam virar fórmulas em planilhas. O PNG tem 1080×1350, fontes locais/fallback e aviso metodológico. Nomes informados aparecem no cartão.

A interface pode ser montada sem abrir servidor. `ABRIR_INTERFACE=False` mantém o notebook executável em testes; para abrir, mude para True na primeira célula ou use `python app.py` no terminal.

In [ ]:
pasta_exportacoes = RAIZ / 'exportacoes_notebook'
csv = exportar_playlist(resultado, pasta_exportacoes / 'playlist_comum.csv')
png = gerar_cartao(resultado, pasta_exportacoes / 'cartao.png')
print('CSV:', csv.name, '| PNG:', png.name)
display(Image(filename=str(png), width=540))

if ABRIR_INTERFACE:
    criar_interface().launch(inbrowser=False, prevent_thread_lock=True)
else:
    print('Interface não iniciada. Execute python app.py quando desejar abrir o aplicativo.')

### Reproduzir a entrega

Na pasta do projeto, com o ambiente instalado:

```bash
python -m pytest -q
python -m ruff check .
python -m ruff format --check .
python tests/test_golden_master.py --escala-grande -v
```

Os benchmarks ficam fora do notebook para não misturar processamento com renderização. Consulte o README para gerar dados de 300 mil eventos e medir medianas com cache frio/quente. Tempos antigos do Passo 2 são registros históricos, não um benchmark dos recursos novos.